# Poverty Map (OpenCellid)

## Setup

### Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!ln -s /content/drive/MyDrive/Colab\ Notebooks/SES-Inference /mydrive

Mounted at /content/drive


### Dependencies

In [ ]:
!pip install geopandas
!pip install pyshp

In [3]:
import os 
import sys
import geopandas as gpd
from shapely.ops import nearest_points
from shapely.strtree import STRtree
from pyproj import Transformer

sys.path.append('/mydrive/libs/')
%set_env PYTHONPATH=/env/python:/mydrive/libs/

%load_ext autoreload
%autoreload 2

from utils import ios
from maps import opencellid
from maps import geo

env: PYTHONPATH=/env/python:/mydrive/libs/


## Data

### Cell IDs

In [4]:
country = 'Uganda'
path = "/mydrive/data/Uganda/connectivity/"
fn_cells = "cell_towers_{}.csv".format(country)

In [5]:
!cp /mydrive/data/OpenCellid/country/'$fn_cells' '$path'

In [6]:
fn_cells = os.path.join(path,fn_cells)
gdf_cells = geo.load_as_GeoDataFrame(fn=fn_cells, index_col=0, lat='lat', lon='lon', crs=geo.PROJ_DEG)
gdf_cells.head()

,radio,mcc,net,area,cell,unit,lon,lat,range,samples,changeable,created,updated,averageSignal,country,geometry
582838,GSM,641,14,1000,11691,0,32.573776,0.315170,1000,1,1,1459743985,1459743985,0,Uganda,POINT (32.57378 0.31517)
205466,UMTS,641,10,6008,132064,0,32.550545,0.403290,1000,12,1,1366333441,1376620582,0,Uganda,POINT (32.55054 0.40329)
205468,UMTS,641,10,6008,147155,0,32.554521,0.402593,1787,47,1,1366334367,1454128236,0,Uganda,POINT (32.55452 0.40259)
210439,GSM,641,10,110,996,0,32.550430,0.403061,1000,1,1,1366548464,1366548464,0,Uganda,POINT (32.55043 0.40306)
210440,GSM,641,10,110,16083,0,32.550430,0.403061,1000,1,1,1366548508,1366548508,0,Uganda,POINT (32.55043 0.40306)


### Survey Data (clusters)

In [8]:
fn_dhs = '/mydrive/data/Uganda/results/DHS2016_MIS2018_iwi_cluster_NTLL.csv'
df_dhs = ios.load_csv(fn_dhs, index_col=0)
gdf_dhs = geo.get_GeoDataFrame(df=df_dhs, lat='LATNUM', lon='LONGNUM', crs=geo.PROJ_DEG)
gdf_dhs.head()

,DHSCC,DHSYEAR,DHSCLUST,URBAN_RURA,LATNUM,LONGNUM,SOURCE,ALT_GPS,ALT_DEM,DATUM,mean_iwi,SES,geometry
0,UG,2016,1,1,0.320188,32.568206,GPS,9999.0,1197.0,WGS84,43.300000,rich,POINT (32.56821 0.32019)
1,UG,2016,2,1,0.340653,32.593627,GPS,9999.0,1179.0,WGS84,24.967857,rich,POINT (32.59363 0.34065)
2,UG,2016,3,1,0.313103,32.566556,GPS,9999.0,1189.0,WGS84,34.732000,rich,POINT (32.56656 0.31310)
3,UG,2016,4,1,0.353368,32.558144,GPS,9999.0,1181.0,WGS84,36.060714,rich,POINT (32.55814 0.35337)
4,UG,2016,5,1,0.367388,32.594357,GPS,9999.0,1226.0,WGS84,44.857692,rich,POINT (32.59436 0.36739)


### Merging

In [9]:
### Projection to meters
gdf_cells_proj = geo.get_projection(gdf_cells, geo.PROJ_MET)
gdf_dhs_proj = geo.get_projection(gdf_dhs, geo.PROJ_MET)
all_cells = geo.get_STRtree(gdf_cells_proj)

In [11]:
### Cluster antennas by similar location (based on given distance)
gdf_cells_proj = geo.fast_identify_clusters_within_distance(gdf_cells_proj, max_distance=5.0, n_jobs=10)
gdf_cells_proj.rename(columns={'CLUSTER_ID':'tower_id'}, inplace=True)

In [12]:
# number of cell ids within cluster
# distance to the closest cell
counter = 5
METERS = [geo.MILE_TO_M, geo.KM_TO_M*2, geo.KM_TO_M*5, geo.KM_TO_M*10]
# urban (1, 2000), rural (2, 5000)
for index, row in gdf_dhs_proj.iterrows():

  # nearest place
  nearest, distance_nearest = geo.find_nearest_place(all_cells, row.geometry)

  # number of cells and towers
  cells_in_area = {}
  towers_in_area = {}
  for meters in METERS:
    k = round(meters/1000.,1)
    cells_area = geo.find_places_sqm(gdf_proj=gdf_cells_proj, point_proj=row.geometry, width=meters)
    cells = cells_area.shape[0]
    towers = cells_area.tower_id.nunique()
    cells_in_area[k] = cells
    towers_in_area[k] = towers
    
  df_dhs.loc[index,'distance_closest_cell'] = distance_nearest
  for k,v in cells_in_area.items():
    df_dhs.loc[index,'cells_in_{}km'.format(k)] = v
  for k,v in towers_in_area.items():
    df_dhs.loc[index,'towers_in_{}km'.format(k)] = v

  print(gdf_dhs.loc[index,'geometry'], cells, towers, distance_nearest, geo.convert(nearest, geo.PROJ_MET, geo.PROJ_DEG))
  counter -= 1
  if counter <= 0:
    break

POINT (32.5682063677 0.320187631511) 8089 5196 81.09421796065045 POINT (32.568932 0.3202519999999999)
POINT (32.5936271164 0.340653350002) 9061 5955 97.16317001271189 POINT (32.593002319336 0.3412628173828099)
POINT (32.5665559657 0.313103244185) 7807 4960 85.6637797781728 POINT (32.566223 0.313797)
POINT (32.5581439634 0.353368071333) 6371 4138 192.10056976072136 POINT (32.557087 0.352004)
POINT (32.5943566295 0.3673876862129999) 5244 3893 14.04604943258065 POINT (32.594261 0.36747)


In [13]:
df_dhs.drop(columns="geometry").head()

,DHSCC,DHSYEAR,DHSCLUST,URBAN_RURA,LATNUM,LONGNUM,SOURCE,ALT_GPS,ALT_DEM,DATUM,mean_iwi,SES,distance_closest_cell,cells_in_1.6km,cells_in_2.0km,cells_in_5.0km,cells_in_10.0km,towers_in_1.6km,towers_in_2.0km,towers_in_5.0km,towers_in_10.0km
0,UG,2016,1,1,0.320188,32.568206,GPS,9999.0,1197.0,WGS84,43.300000,rich,81.094218,625.0,1022.0,4121.0,8089.0,322.0,516.0,2307.0,5196.0
1,UG,2016,2,1,0.340653,32.593627,GPS,9999.0,1179.0,WGS84,24.967857,rich,97.163170,383.0,596.0,3133.0,9061.0,229.0,350.0,2144.0,5955.0
2,UG,2016,3,1,0.313103,32.566556,GPS,9999.0,1189.0,WGS84,34.732000,rich,85.663780,500.0,766.0,3741.0,7807.0,242.0,377.0,2081.0,4960.0
3,UG,2016,4,1,0.353368,32.558144,GPS,9999.0,1181.0,WGS84,36.060714,rich,192.100570,140.0,198.0,1291.0,6371.0,125.0,177.0,1029.0,4138.0
4,UG,2016,5,1,0.367388,32.594357,GPS,9999.0,1226.0,WGS84,44.857692,rich,14.046049,118.0,174.0,1179.0,5244.0,92.0,143.0,949.0,3893.0


## All-in-one

In [14]:
df_dhs_new = opencellid.update_opencellid_features(fn_cells, fn_dhs, antenna_same_tower_max_distance=5.0)
df_dhs_new.head()

100%|██████████| 1001/1001 [39:25<00:00,  2.36s/it]


,mean_iwi,SES,distance_closest_cell,cells_in_1.6km,cells_in_2.0km,cells_in_5.0km,cells_in_10.0km,towers_in_1.6km,towers_in_2.0km,towers_in_5.0km,towers_in_10.0km
0,43.300000,rich,81.094218,625.0,1022.0,4121.0,8089.0,322.0,516.0,2307.0,5196.0
1,24.967857,rich,97.163170,383.0,596.0,3133.0,9061.0,229.0,350.0,2144.0,5955.0
2,34.732000,rich,85.663780,500.0,766.0,3741.0,7807.0,242.0,377.0,2081.0,4960.0
3,36.060714,rich,192.100570,140.0,198.0,1291.0,6371.0,125.0,177.0,1029.0,4138.0
4,44.857692,rich,14.046049,118.0,174.0,1179.0,5244.0,92.0,143.0,949.0,3893.0
